In [1]:
import json
import torch
import torch.nn.functional as F
import numpy as np

from transformers import AutoTokenizer, AutoModel

/opt/conda/envs/minicpmv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

In [3]:
def compute_similarity(model, tokenizer, file_path1, file_path2, device):
    with open(file_path1, 'r') as f:
        data1 = json.load(f)
    with open(file_path2, 'r') as f:
        data2 = json.load(f)
    
    total_score = 0.0
    total_cnt = 0
    for idx, original_entry in enumerate(data1):
        unlearning_entry = data2[idx]
        
        assert original_entry['id'] == unlearning_entry['id'], "id not match"
        
        for i, conversation in enumerate(original_entry['conversations']):
            if conversation['role'] == 'assistant':
                original_sentence = conversation['original_model_answer']
                unlearning_sentence = unlearning_entry['conversations'][i]['answer']
                sentences = [original_sentence, unlearning_sentence]
                encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt').to(device)
                
                with torch.no_grad():
                    model_output = model(**encoded_input)
                sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
                sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
                similarity = F.cosine_similarity(sentence_embeddings[0], sentence_embeddings[1], dim=0)
                total_score += similarity.item()
                total_cnt += 1
        print(f"Finish {idx+1}/{len(data1)}")
                
    return 1.0 - total_score / total_cnt

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("../../checkpoints/all-mpnet-base-v2")
model = AutoModel.from_pretrained("../../checkpoints/all-mpnet-base-v2").to(device)

/opt/conda/envs/minicpmv/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [ ]:
file_path1 = "../../data/vqa/FIOFP/FIOFP_100_with_original_output.json"
file_path2 = "../../output/model_output/emmu_model/FIOFP_100_output.json"

similarity = compute_similarity(model, tokenizer, file_path1, file_path2, device)
print(f"Similarity score: {similarity}")